In [3]:
import pandas as pd
import numpy as np

# =========================
# 1. 파일 불러오기
# =========================
INPUT_FILE = "통합데이터.xlsx"
OUTPUT_FILE = "정합성점검_결과.xlsx"

df = pd.read_excel(INPUT_FILE)

# =========================
# 2. 컬럼명/문자열 정리
# =========================
def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = " ".join(x.split())
    return x if x != "" else np.nan

df.columns = [str(c).strip() for c in df.columns]

text_cols = [
    "소재지", "지목", "토지이동사유", "토지이동일", "상태",
    "고시지목", "토지지목", "출자지목"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# =========================
# 3. 숫자형 변환
# =========================
def to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "").strip()
    if s == "":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

num_cols = ["면적", "고시면적", "토지면적", "출자면적"]

for col in num_cols:
    if col in df.columns:
        df[col] = df[col].apply(to_float)

# =========================
# 4. 비교 함수
# =========================
def area_diff(base, target):
    """
    면적 비교
    - 둘 중 하나라도 결측이면 비교 제외(False)
    - 둘 다 값이 있을 때만 비교
    """
    if pd.isna(base) or pd.isna(target):
        return False
    return base != target

def category_diff(base, target):
    """
    지목 비교
    - 둘 중 하나라도 결측이면 비교 제외(False)
    - 둘 다 값이 있을 때만 비교
    """
    if pd.isna(base) or pd.isna(target):
        return False
    return str(base).strip() != str(target).strip()

def make_flag(area_is_diff, cat_is_diff):
    if area_is_diff and cat_is_diff:
        return 3
    elif area_is_diff:
        return 1
    elif cat_is_diff:
        return 2
    else:
        return 0

# =========================
# 5. 비교 시트용 기타값 생성
# =========================
# 1번 시트: 고시비교
df["고시_면적불일치"] = df.apply(lambda r: area_diff(r["면적"], r["고시면적"]), axis=1)
df["고시_지목불일치"] = df.apply(lambda r: category_diff(r["지목"], r["고시지목"]), axis=1)
df["고시_기타"] = df.apply(lambda r: make_flag(r["고시_면적불일치"], r["고시_지목불일치"]), axis=1)

# 2번 시트: 항만비교(=토지비교)
df["항만_면적불일치"] = df.apply(lambda r: area_diff(r["면적"], r["토지면적"]), axis=1)
df["항만_지목불일치"] = df.apply(lambda r: category_diff(r["지목"], r["토지지목"]), axis=1)
df["항만_기타"] = df.apply(lambda r: make_flag(r["항만_면적불일치"], r["항만_지목불일치"]), axis=1)

# 3번 시트: 출자비교
df["출자_면적불일치"] = df.apply(lambda r: area_diff(r["면적"], r["출자면적"]), axis=1)
df["출자_지목불일치"] = df.apply(lambda r: category_diff(r["지목"], r["출자지목"]), axis=1)
df["출자_기타"] = df.apply(lambda r: make_flag(r["출자_면적불일치"], r["출자_지목불일치"]), axis=1)

# =========================
# 6. 말소여부 / 등재현황 로직
# =========================
def get_malso(row):
    status = "" if pd.isna(row.get("상태")) else str(row.get("상태"))
    reason = "" if pd.isna(row.get("토지이동사유")) else str(row.get("토지이동사유"))
    if ("말소" in status) or ("말소" in reason):
        return "말소"
    return "정상"

def exists_gosi(row):
    return (
        pd.notna(row.get("고시면적")) or
        pd.notna(row.get("고시지목"))
    )

def exists_hangman(row):
    return (
        pd.notna(row.get("토지면적")) or
        pd.notna(row.get("토지지목"))
    )

def exists_chulja(row):
    return (
        pd.notna(row.get("출자면적")) or
        pd.notna(row.get("출자지목"))
    )

def get_register_status(row):
    in_port = exists_hangman(row)
    in_invest = exists_chulja(row)

    if in_port and in_invest:
        return "항만·출자 등재"
    elif in_port:
        return "항만 등재"
    elif in_invest:
        return "출자 등재"
    else:
        return "미등재"

df["말소여부"] = df.apply(get_malso, axis=1)
df["고시존재"] = df.apply(exists_gosi, axis=1)
df["항만존재"] = df.apply(exists_hangman, axis=1)
df["출자존재"] = df.apply(exists_chulja, axis=1)
df["등재현황"] = df.apply(get_register_status, axis=1)

# =========================
# 7. 각 시트 생성
# =========================
# 1번시트(고시비교)
sheet1 = df[[
    "소재지", "면적", "고시면적", "지목", "고시지목"
]].copy()
sheet1["기타"] = df["고시_기타"]

# 2번시트(항만비교)
sheet2 = df[[
    "소재지", "면적", "토지면적", "지목", "토지지목"
]].copy()
sheet2 = sheet2.rename(columns={
    "토지면적": "항만면적",
    "토지지목": "항만지목"
})
sheet2["기타"] = df["항만_기타"]

# 3번시트(출자비교)
sheet3 = df[[
    "소재지", "면적", "출자면적", "지목", "출자지목"
]].copy()
sheet3["기타"] = df["출자_기타"]

# 4번시트(소재지점검)
# 고시에는 없는데 항만 또는 출자에는 있는 것
sheet4 = df[
    (~df["고시존재"]) & (df["항만존재"] | df["출자존재"])
][[
    "소재지", "말소여부", "등재현황"
]].copy()

# 5번시트(면적불일치)
# 세 비교 중 하나라도 면적 불일치면 포함
sheet5 = df[
    df["고시_면적불일치"] | df["항만_면적불일치"] | df["출자_면적불일치"]
][[
    "소재지", "면적", "고시면적", "토지면적", "출자면적"
]].copy()
sheet5 = sheet5.rename(columns={"토지면적": "항만면적"})

# 6번시트(지목불일치)
# 세 비교 중 하나라도 지목 불일치면 포함
sheet6 = df[
    df["고시_지목불일치"] | df["항만_지목불일치"] | df["출자_지목불일치"]
][[
    "소재지", "지목", "고시지목", "토지지목", "출자지목"
]].copy()
sheet6 = sheet6.rename(columns={"토지지목": "항만지목"})

# =========================
# 8. 정렬(선택)
# =========================
for s in [sheet1, sheet2, sheet3, sheet4, sheet5, sheet6]:
    if "소재지" in s.columns:
        s.sort_values("소재지", inplace=True)

# =========================
# 9. 엑셀 저장
# =========================
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    sheet1.to_excel(writer, sheet_name="1_고시비교", index=False)
    sheet2.to_excel(writer, sheet_name="2_항만비교", index=False)
    sheet3.to_excel(writer, sheet_name="3_출자비교", index=False)
    sheet4.to_excel(writer, sheet_name="4_소재지점검", index=False)
    sheet5.to_excel(writer, sheet_name="5_면적불일치", index=False)
    sheet6.to_excel(writer, sheet_name="6_지목불일치", index=False)

print(f"완료: {OUTPUT_FILE}")
print()
print("고시비교 기타 분포")
print(sheet1["기타"].value_counts(dropna=False).sort_index())
print()
print("항만비교 기타 분포")
print(sheet2["기타"].value_counts(dropna=False).sort_index())
print()
print("출자비교 기타 분포")
print(sheet3["기타"].value_counts(dropna=False).sort_index())
print()
print("소재지점검 건수:", len(sheet4))
print("면적불일치 건수:", len(sheet5))
print("지목불일치 건수:", len(sheet6))

완료: 정합성점검_결과.xlsx

고시비교 기타 분포
기타
0    349
1     18
2     23
3      3
Name: count, dtype: int64

항만비교 기타 분포
기타
0    350
1     26
2     15
3      2
Name: count, dtype: int64

출자비교 기타 분포
기타
0    319
1     54
2     14
3      6
Name: count, dtype: int64

소재지점검 건수: 35
면적불일치 건수: 77
지목불일치 건수: 39
